# Examine Relationship Between Combine Features And Separation During Go/Post/Corner Routes

**<span style="color: gold;">PROJECT</span>**

WR/DB Separation Analysis

**<span style="color: gold;">GOAL OF THIS NOTEBOOK</span>**

We've previously shown that there is a significant relationship between our Combine metrics (more specifically, our deceleration score) and separation after the cut during cutting routes (Hitch, In, Out routes).

We will now expand this to examine if there's a relationship between our Combine metrics and creating separation in long routes that don't require cutting (Go, Post, Corner routes).

We hypothesize that our Combine features, more specifically our acceleration and speed features will be associated with increased separation in these long, cutless routes.

**<span style="color: gold;">FINDINGS</span>**

There appears to be a significant relationship between *accel_score* and *avg_go_separation_generated*. If a player has a higher acceleration score, then they tend to create more seperation during their burst on Go/Post/Corner routes. This stands even when adjusting for top speed.

* p = 0.048, R-squared = 0.288, N = 14
* *when controlling for general agility (from master_agility_score)*: p = 0.083, R-squared = 0.33, N = 11

**<span style="color: gold;">NOTES</span>**

In this analysis, we will be derive separation metrics that use WR player-tracking data and player-tracking data for their defender. Due to our need for adequate defender data, the 2027 Big Data Bowl data will not suffice. Therefore, we will use the 2026 Big Data Bowl data, which means that we can only examine 2023 rookies.


<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    BDB_2026_BEFORE_PASS_DIR,
    SEPARATION_DATA_DIR,
)

import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"
from datetime import datetime

import statsmodels.api as sm

import sys
from pathlib import Path

# ==========================================
# Identify Project Root
# ==========================================
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)

# ==========================================
# Import Shared Data Schemas
# ==========================================
sys.path.insert(0, str(PROJECT_ROOT))
from utils.schema import schema_map

<br><br/>
## Load Data
***

#### Set Parameters For Analysis

In [ ]:
# =======================================
# Import Parameters That Define What
# Data We'll Examine In This Project
# =======================================

param_path = PROJECT_ROOT / "05_WR_Separation_Analysis" / "Data_Pull"
sys.path.insert(0, str(param_path))

import Data_Parameters as params

In [ ]:
# =======================================
# Specify Routes of Interest
# =======================================

ROUTES_OF_INTEREST = ["GO", "CORNER", "POST"]

<br><br/>
#### Play and Player Level Details

In [ ]:
# ===========================================
# Player-Time Level WR/DB Player-Tracking & Play Data
# ===========================================

play_player_details_df = pl.read_csv(str(SEPARATION_DATA_DIR / "00_2023_Rookie_Player_Time_Level_Play_Details_Data.csv"))

print("Play/Player Details Data:")
print(f"    - Number of Unique WRs: {play_player_details_df.filter(pl.col("player_position") == "WR").select("nfl_id").n_unique()}")
print(f"    - Number of WR Player-Plays: {play_player_details_df.filter(pl.col("player_position") == "WR").select("player_play_id").n_unique()}")
play_player_details_df

<br><br/>
#### WR Separation Data From *01_BDB26_WR_Player_Tracking.ipynb*

In [ ]:
# =========================================
# WR Separation Data Among 2023 Rookies
# =========================================

wr_separation_df = pl.read_csv(str(SEPARATION_DATA_DIR / "01_2023_Rookie_Player_Time_Level_Separation_Data.csv"))

# -----------------------------------------
# Identify Players Of Interest
# -----------------------------------------

players_of_interest = (
    wr_separation_df
    .select("wr_nfl_id")
    .unique()
    .to_series()
    .to_list()
)

print("Separation Data:")
print(f"    - Dimensions: {wr_separation_df.shape}")
print(f"    - Number of Rookie WRs: {wr_separation_df.select("wr_nfl_id").n_unique()}")
print(f"    - Number of Player-Plays: {wr_separation_df.select("player_play_id").n_unique()}")
wr_separation_df

<br><br/>
#### Combine Features Data

In [ ]:
# =======================================
# Combine Features Data
# =======================================

combine_features_df = pl.read_csv(str(SEPARATION_DATA_DIR / "03_WR_Combine_Features_Data.csv"))

print("Combine Features Data:")
print(f"    - Dimensions: {combine_features_df.shape}")
print(f"    - Number of Players: {combine_features_df.select("nfl_id").n_unique()}")
combine_features_df

<br><br/>
## Data Cleaning
***

#### Filter Data Based On Inclusion Criteria

In [ ]:
# ===============================
# Number of WRs In WR Separation Data
# ===============================
wr_player_plays = wr_separation_df.select("player_play_id").unique().to_series().to_list()
print("WRs In WR Separation Data:")
print(f"    - Number of Unique WR Player-Plays: {len(wr_player_plays)}")

In [ ]:
# ===============================
# How Many of Those Player-Plays Do We Have Route Data For?
# ===============================

# -------------------------------
# For BDB2027 Route Data, Use route_ran
# -------------------------------

bdb2027_extra_route_data = (
    play_player_details_df
    .filter( 
        pl.col("route_ran").is_not_null()
    )
    .group_by("player_play_id")
    .agg(
        route_ran = pl.col("route_ran").first()
    )
)

# -------------------------------
# For BDB2026 Route Data For Targeted 
# Receivers, Use route_of_targetered_receiver
# -------------------------------

target_routes = (
    play_player_details_df
    .filter(
        pl.col("target"), 
        pl.col("route_of_targeted_receiver").is_not_null()
    )
    .group_by("player_play_id")
    .agg(
        route_of_targeted_receiver = pl.col("route_of_targeted_receiver").first()
    )
)

# -------------------------------
# Target Data
# -------------------------------

target_data = (
    play_player_details_df
    .select("player_play_id", "target")
    .unique()
)

# -------------------------------
# Add Routes To WR Separation Data
# -------------------------------

wr_separation_df = (
    wr_separation_df
    # Add target data
    .join(
        target_data, 
        on = "player_play_id", 
        how = "inner"
    )
    # Add non-targeted routes
    .join(
        bdb2027_extra_route_data, 
        on = "player_play_id", 
        how = "left"
    )
    # Add targeted routes
    .join(
        target_routes, 
        on = "player_play_id", 
        how = "left"
    )
    # Create route variable
    .with_columns(
        route = (
            # If target and has route_of_targeted_receiver data, then use that
            pl.when((pl.col("target") == True) & (pl.col("route_of_targeted_receiver").is_not_null()))
              .then(pl.col("route_of_targeted_receiver"))
            # Otherwise, use route_ran
              .otherwise(pl.col("route_ran"))
        )
    )
)

# -----------------------------
# Get Rid Of Player-Plays With No Route Data
# -----------------------------

wr_separation_df = (
    wr_separation_df
    .filter(pl.col("route").is_not_null())
)

# -----------------------------
# Only Consider Targeted Receivers
# -----------------------------

wr_separation_df = (
    wr_separation_df.filter("target")
)

# -----------------------------
# Only Consider These Route Types
# -----------------------------

wr_separation_df = (
    wr_separation_df
    .filter(pl.col("route").is_in(ROUTES_OF_INTEREST))
)

# -----------------------------
# Only Consider Man Coverage
# -----------------------------

wr_separation_df = (
    wr_separation_df
    # -------------------------------
    # Add team_coverage_man_zone
    # -------------------------------
    .join(
        # Get the coverage for each play
        play_player_details_df
        .select("game_play_id", "team_coverage_man_zone")
        # Add .unique() to make sure we don't have duplicate rows!
        .unique(), 
        on = "game_play_id",
        how = "inner"
    )
    # -------------------------------
    # Filter To Man Coverage
    # -------------------------------
    .filter(pl.col("team_coverage_man_zone") == "MAN_COVERAGE")
)


print(f"Updated Assignments Data:")
print(f"    - Number of WR Player-Plays: {wr_separation_df.select("player_play_id").n_unique()}")
print(f"    - Number of WRs: {wr_separation_df.select("wr_nfl_id").n_unique()}")
print(f"    - Number of Unique Plays: {wr_separation_df.select("game_play_id").n_unique()}")
wr_separation_df

<br><br/>
#### Add *absolute_yardline_number* To WR Separation Data

In [ ]:
# ===============================================
# Add absolute_yardline_number To wr_separation_df
# ===============================================

data_dir = Path(str(BDB_2026_BEFORE_PASS_DIR))

# ------------------------------------
# Load and Combine 18 Before Pass Datasets Into 1
# ------------------------------------

absolute_yardline_number_df = (
    pl.concat(
        [
            pl.scan_csv(data_dir / f"input_2023_w{i:02d}.csv")
            .with_columns(pl.lit(i).alias("week"))
            for i in range(1, 19)
        ],
        how="vertical"
    )
    .collect()
)

# ------------------------------------
# Add Game-Play and Player-Play ID
# ------------------------------------

absolute_yardline_number_df = (
    absolute_yardline_number_df
    .with_columns(game_play_id = pl.col("game_id").cast(pl.String) + "_" + pl.col("play_id").cast(pl.String))
    .group_by("game_play_id")
    .agg(absolute_yardline_number = pl.col("absolute_yardline_number").first())
)

absolute_yardline_number_df

In [ ]:
wr_separation_df = (
    wr_separation_df
    .join(
        absolute_yardline_number_df, 
        on = "game_play_id", 
        how = "inner"
    )
)

wr_separation_df

In [ ]:
# =======================================
# Num Player-Plays Per Player
# =======================================

(
    wr_separation_df
    .group_by("wr_nfl_id")
    .agg(num_player_plays = pl.col("player_play_id").n_unique())
    .sort("num_player_plays", descending = True)
)



<br><br/>
#### Create Metrics For Separation In Routes

In [ ]:
# ==============================================================================
# Extract In-Game Acceleration and Separation for Go Routes
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. Calculate Downfield Kinematics and Filter for Vertical Push
# ------------------------------------------------------------------------------
go_kinematics_df = (
    wr_separation_df
    .sort(["player_play_id", "frame_id"])
    .with_columns(
        # Calculate raw directional velocities (dt = 0.1s at 10Hz)
        vx = (pl.col("wr_x") - pl.col("wr_x").shift(1).over("player_play_id")) / 0.1,
        vy = (pl.col("wr_y") - pl.col("wr_y").shift(1).over("player_play_id")) / 0.1,
    )
    .with_columns(
        # Speed magnitude
        s = (pl.col("vx")**2 + pl.col("vy")**2).sqrt()
    )
    .with_columns(
        # True linear acceleration
        linear_accel = (pl.col("s") - pl.col("s").shift(1).over("player_play_id")) / 0.1
    )
    .with_columns(
        # Isolate true downfield velocity based on play direction
        v_downfield = pl.when(pl.col("play_direction") == "right")
                      .then(pl.col("vx"))
                      .otherwise(-pl.col("vx")),
        # Calculate relative distance from the Line of Scrimmage (LOS)
        # Assumes absolute distance downfield from where the play started
        yards_midfield = pl.when(pl.col("play_direction") == "right")
                         .then(pl.col("wr_x") - pl.col("absolute_yardline_number"))
                         .otherwise(pl.col("absolute_yardline_number") - pl.col("wr_x"))
    )
)

# ------------------------------------------------------------------------------
# 2. Isolate Key Frames: The Snap (Release) and Peak Downfield Burst
# ------------------------------------------------------------------------------
# For Go routes, we want to find the frame where they hit their absolute highest 
# acceleration gear *after* they've advanced at least 5 yards past the LOS. 
# This prevents capturing short hand-fights at the line and focuses on deep vertical burn.
go_burst_frames_df = (
    go_kinematics_df
    .filter(
        (pl.col("linear_accel").is_not_null()) & 
        (pl.col("yards_midfield") > 5.0) # Clear the initial line-of-scrimmage jam
    )
    .group_by("player_play_id")
    .agg([
        # Find the frame where the WR hits maximum acceleration downfield
        pl.col("frame_id").filter(pl.col("linear_accel") == pl.col("linear_accel").max()).first().alias("peak_burst_frame_id"),
        # Also capture the snap/start frame of the play for baseline release tracking
        pl.col("frame_id").min().alias("snap_frame_id")
    ])
)

# ------------------------------------------------------------------------------
# 3. Extract Separation at Release and Peak Vertical Burst
# ------------------------------------------------------------------------------
final_go_separation_df = (
    go_kinematics_df
    .join(go_burst_frames_df, on="player_play_id", how="inner")
    .filter(
        (pl.col("frame_id") == pl.col("snap_frame_id")) |                          # At the line
        (pl.col("frame_id") == (pl.col("snap_frame_id") + 10)) |                    # 1.0s after snap (Release Separation)
        (pl.col("frame_id") == pl.col("peak_burst_frame_id")) |                    # Moment of peak deep acceleration
        (pl.col("frame_id") == (pl.col("peak_burst_frame_id") + 5))                 # 0.5s after peak acceleration
    )
    .group_by("player_play_id")
    .agg([
        pl.col("wr_nfl_id").first(),
        pl.col("wr_name").first(),
        pl.col("route").first(),
        
        # 1. Initial Release Separation: How well they stack the CB in the first 10 frames (1 second)
        pl.col("wr_cb_distance").filter(pl.col("frame_id") == (pl.col("snap_frame_id") + 10)).first().alias("separation_at_release"),
        
        # 2. Deep Burst Separation: Distance to CB the exact moment they turn on their highest gear
        pl.col("wr_cb_distance").filter(pl.col("frame_id") == pl.col("peak_burst_frame_id")).first().alias("separation_at_peak_burst"),
        
        # 3. Post-Burst Separation: Did the acceleration actually burn the CB 0.5 seconds later?
        pl.col("wr_cb_distance").filter(pl.col("frame_id") == (pl.col("peak_burst_frame_id") + 5)).first().alias("separation_post_burst")
    ])
    .with_columns(
        # The ultimate metric: Net separation generated *by* the downfield burst
        go_separation_generated = pl.col("separation_post_burst") - pl.col("separation_at_peak_burst")
    )
)

# ------------------------------------------------------------------------------
# 4. Roll up to Player Averages
# ------------------------------------------------------------------------------
player_go_metrics_df = (
    final_go_separation_df
    .group_by("wr_nfl_id")
    .agg([
        pl.col("wr_name").first(),
        pl.col("separation_at_release").mean().alias("avg_release_separation"),
        pl.col("separation_at_peak_burst").mean().alias("avg_sep_at_burst_point"),
        pl.col("go_separation_generated").mean().alias("avg_go_separation_generated")
    ])
)

player_go_metrics_df

<br><br/>
#### Add Combine Metrics To Separation Data

In [ ]:
analysis_df = (
    player_go_metrics_df
    .join(
        combine_features_df, 
        left_on="wr_nfl_id", 
        right_on="nfl_id", 
        how = "inner"
    )
)

print("Final Analysis Dataset:")
print(f"    - Number of WRs: {analysis_df.select("wr_nfl_id").n_unique()}")
analysis_df

<br><br/>
## Examine Associations
***

#### Acceleration Score vs Separation Generated By Burst

In [ ]:
predictor = "accel_score" 
outcome = "avg_go_separation_generated"

<br><br/>
***Scatter Plot***

In [ ]:
predictor = "accel_score" 
outcome = "avg_go_separation_generated"

fig = px.scatter(
    data_frame = analysis_df, 
    x = predictor, 
    y = outcome, 
    trendline = "ols", 
    title = "Acceleration Score vs Average Separation Created By Burst", 
    labels = {
        "accel_score": "Acceleration Score", 
        "avg_go_separation_generated": "Average Separation Created (yards)"
    },
    template = "simple_white"
)

# fig.update_xaxes(
#     range = [-0.5, analysis_df["accel_score"].max() + 2]
# )

fig.update_traces(
    line=dict(color="grey"),
    selector=dict(mode="lines")
)

fig.update_traces(
    marker=dict(size=10),
    selector=dict(mode="markers")
)

fig.show()

<br><br/>
***Univariate Model***

In [ ]:
# ==========================================
# Univariate Model
# ==========================================

regression_df = analysis_df

regression_df = regression_df.filter(
    (pl.col(predictor).is_not_null())
    & 
    (pl.col(outcome).is_not_null())

)

pd_reg = regression_df.to_pandas()

X = pd_reg[predictor]
y = pd_reg[outcome]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

<br><br/>
***Leave-One-Out Analysis***

In [ ]:
# ==========================================
# Leave-One-Out Sensitivity Analysis
# ==========================================

# ----------------------------------------
# Calculate LOO Metrics
# ----------------------------------------

# 1. Prepare regression data
regression_df = (
    analysis_df
    .filter(
        pl.col(predictor).is_not_null()
        & pl.col(outcome).is_not_null()
    )
)

pd_reg = regression_df.to_pandas().reset_index(drop=True)

n = len(pd_reg)

# Check sample size
print(f"Sample Size: {n}")

# 2. Fit a model leaving out each observation
loo_results = []

for i in range(n):

    # Remove one observation
    loo_df = pd_reg.drop(index=i)

    # Define predictor and outcome
    X = sm.add_constant(loo_df[[predictor]])
    y = loo_df[outcome]

    # Fit OLS model
    model = sm.OLS(y, X).fit()

    # Store results for the predictor (excluding intercept)
    loo_results.append({
        "omitted_observation": i,
        "coef": model.params[predictor],
        "p_value": model.pvalues[predictor],
        "r_squared": model.rsquared,
    })

# 3. Create results table
loo_results_df = pd.DataFrame(loo_results)

# ---------------------------------------
# Scatter Plot of P-Values
# ---------------------------------------

fig = px.line(
    data_frame = loo_results_df, 
    x = "omitted_observation", 
    y = "p_value", 
    labels = {
        "p_value": "P-Value", 
        "omitted_observation": "Iteration"
    },
    title = "P-Values From Leave-One-Out Analysis", 
    template = "simple_white"
)

fig.update_yaxes(
    range = [0, loo_results_df["p_value"].max() + 0.01]
)

fig.add_hline(
    y = 0.10, 
    line_dash = "dash", 
    line_color = "red", 
    line_width = 2
)

fig.show()

In [ ]:
# ==========================================
# Leave-One-Out Sensitivity Analysis (cont'd)
# ==========================================

# ------------------------------------------
# Full Model
# ------------------------------------------

X_full = sm.add_constant(pd_reg[[predictor]])
y_full = pd_reg[outcome]

full_model = sm.OLS(y_full, X_full).fit()

full_results = {
    "coef": full_model.params[predictor],
    "p_value": full_model.pvalues[predictor],
    "r_squared": full_model.rsquared,
}

# ------------------------------------------
# Leave-One-Out Summary
# ------------------------------------------

summary_df = pd.DataFrame({
    "Full Model": full_results,
    "LOO Mean": loo_results_df[
        ["coef", "p_value", "r_squared"]
    ].mean(),
    "LOO Minimum": loo_results_df[
        ["coef", "p_value", "r_squared"]
    ].min(),
    "LOO Maximum": loo_results_df[
        ["coef", "p_value", "r_squared"]
    ].max(),
})

display(summary_df.round(2))

<br><br/>
***Multivariate Model***

In [ ]:
# ===========================================
# Multiple Linear Regression
# ===========================================

# Does the relationship between accel_score and avg_go_separation_generated
# hold if we adjust for general agility (master_agility_score)?

predictors = [
    "master_agility_score",
    "accel_score"
]

corr = round(analysis_df.select(pl.corr(predictors[0], predictors[1])).item(), 2)
vif = 1 / (1 - (corr**2))
print(f"Correlation Between {predictors[0]} and {predictors[1]}: {corr} (VIF = {round(vif, 2)})\n")

regression_df = analysis_df.filter(
    pl.all_horizontal(
        [pl.col(col).is_not_null() for col in predictors + [outcome]]
    )
)

pd_reg = regression_df.to_pandas()

X = pd_reg[predictors]
y = pd_reg[outcome]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

<br><br/>
***Adjust For Route Type***

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ==============================================================================
# 1. Prepare Player-Level Data
# ==============================================================================

predictor = "accel_score"
outcome = "avg_go_separation_generated"
player_id = "wr_nfl_id"

player_df = (
    analysis_df
    .filter(
        pl.col(predictor).is_not_null()
        & pl.col(outcome).is_not_null()
    )
    .select([player_id, predictor, outcome])
    .to_pandas()
)

# Ensure one row per player
assert player_df[player_id].is_unique, (
    "analysis_df must contain one row per WR."
)

eligible_players = player_df[player_id].unique()

print(f"Eligible players: {len(eligible_players)}")


# ==============================================================================
# 2. Baseline Model: Player-Level OLS
# ==============================================================================

baseline_model = smf.ols(
    formula=f"{outcome} ~ {predictor}",
    data=player_df,
).fit()

baseline_coef = baseline_model.params[predictor]
baseline_se = baseline_model.bse[predictor]
baseline_p = baseline_model.pvalues[predictor]
baseline_ci = baseline_model.conf_int().loc[predictor]


# ==============================================================================
# 3. Prepare Play-Level Data for Mixed-Effects Model
# ==============================================================================

play_df = (
    final_go_separation_df
    .filter(
        pl.col(player_id).is_in(eligible_players.tolist())
        & pl.col("separation_post_burst").is_not_null()
        & pl.col("route").is_in(ROUTES_OF_INTEREST)
    )
    .select([
        "player_play_id",
        player_id,
        "route",
        "separation_post_burst",
    ])
    .to_pandas()
)

# Confirm one observation per player-play
assert play_df["player_play_id"].is_unique, (
    "Expected one row per player-play."
)

# Attach each player's deceleration score
play_df = play_df.merge(
    player_df[[player_id, predictor]],
    on=player_id,
    how="inner",
    validate="many_to_one",
)

# Remove any remaining missing values
play_df = play_df.dropna(
    subset=[
        "separation_post_burst",
        predictor,
        "route",
        player_id,
    ]
).copy()

play_df["route"] = play_df["route"].astype("category")

# Check actual sample after filtering
print(f"Players in mixed model: {play_df[player_id].nunique()}")
print(f"Player-plays in mixed model: {len(play_df)}")

print("\nRoute counts:")
print(play_df["route"].value_counts())



# ==============================================================================
# 4. Mixed-Effects Model
#    Fixed effects: accel_score + route
#    Random intercept: WR
# ==============================================================================

mixed_model = smf.mixedlm(
    formula="separation_post_burst ~ accel_score + C(route)",
    data=play_df,
    groups=play_df[player_id],
    re_formula="1",
)

mixed_fit = mixed_model.fit(
    reml=False,
    method="lbfgs",
    maxiter=2000,
)

print("\nMixed-Effects Model Summary:")
print(mixed_fit.summary())


# ==============================================================================
# 5. Compare the Deceleration Coefficients
# ==============================================================================

mixed_coef = mixed_fit.fe_params[predictor]
mixed_se = mixed_fit.bse_fe[predictor]
mixed_p = mixed_fit.pvalues[predictor]
mixed_ci = mixed_fit.conf_int().loc[predictor]

comparison_df = pd.DataFrame([
    {
        "model": "Baseline OLS",
        "predictor": predictor,
        "coef": baseline_coef,
        "std_error": baseline_se,
        "p_value": baseline_p,
        "ci_lower": baseline_ci.iloc[0],
        "ci_upper": baseline_ci.iloc[1],
        "n_players": int(baseline_model.nobs),
        "n_plays": int(baseline_model.nobs),
        "r_squared": baseline_model.rsquared,
        "AIC": baseline_model.aic,
    },
    {
        "model": "Route-adjusted mixed effects",
        "predictor": predictor,
        "coef": mixed_coef,
        "std_error": mixed_se,
        "p_value": mixed_p,
        "ci_lower": mixed_ci.iloc[0],
        "ci_upper": mixed_ci.iloc[1],
        "n_players": play_df[player_id].nunique(),
        "n_plays": len(play_df),
        "r_squared": np.nan,
        "AIC": mixed_fit.aic,
    },
])

print("\nDeceleration Effect Comparison:")
print(comparison_df.round(4).to_string(index=False))


# ==============================================================================
# 6. Quantify How Much the Deceleration Coefficient Changed
# ==============================================================================

coef_change = mixed_coef - baseline_coef

if baseline_coef != 0:
    pct_change = 100 * (mixed_coef - baseline_coef) / abs(baseline_coef)
else:
    pct_change = np.nan

print(f"\nBaseline coefficient: {baseline_coef:.4f}")
print(f"Route-adjusted coefficient: {mixed_coef:.4f}")
print(f"Absolute coefficient change: {coef_change:.4f}")
print(f"Change relative to baseline magnitude: {pct_change:.1f}%")
print(f"Mixed model converged: {mixed_fit.converged}")


<br><br/>
## Export Separation Data
***

In [ ]:
# =============================================
# Export Separation Data
# =============================================

outcome_variable = "avg_go_separation_generated"

burst_separation_df = (
    analysis_df
    .rename({"wr_nfl_id": "nfl_id"})
    .filter(pl.col(outcome_variable).is_not_null())
    .select("nfl_id", outcome_variable)
)

burst_separation_df.write_csv(str(SEPARATION_DATA_DIR / "22_WR_Bursting_Separation_Data.csv"))